# Prepare LoRA Data

Convert benchmark records into the chat JSONL format expected by `mlx_lm.lora`.

In [51]:
from pathlib import Path
import json
import sys

In [52]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune')

In [53]:
from training_eval.eval_utils import load_jsonl_records

In [54]:
SYSTEM_MESSAGE = 'You solve discrete stochastic-process problems. Give concise reasoning, then end with exactly one final answer block: Final answer:\n<answer>\n{...}\n</answer>. Do not write anything after </answer>.'


In [55]:
DATASETS = {
    "formula_direct": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_formula_direct",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_formula_direct",
    },
    "algorithmic_scaffold": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold",
    },
    "algorithmic_scaffold_v2": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v2",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v2",
    },
    "algorithmic_scaffold_v2_5": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v2_5",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v2_5",
    },
    "algorithmic_scaffold_v3": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3",
    },
    "algorithmic_scaffold_v3_1": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_1",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_1",
    },
    "algorithmic_scaffold_v3_5": {
        "train": PROJECT_ROOT / "benchmark" / "data" / "train_algorithmic_scaffold_v3_5",
        "valid": PROJECT_ROOT / "benchmark" / "data" / "val_algorithmic_scaffold_v3_5",
    },
}

OUTPUT_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"


In [56]:
def make_chat_example(record):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": record["problem"]},
            {"role": "assistant", "content": record["reasoning"]},
        ]
    }

In [57]:
def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        for record in records:
            f.write(json.dumps(record, sort_keys=True) + "\n")

In [58]:
def build_lora_dataset(name, paths):
    train_records = load_jsonl_records(paths["train"])
    valid_records = load_jsonl_records(paths["valid"])
    output_dir = OUTPUT_ROOT / name

    write_jsonl([make_chat_example(record) for record in train_records], output_dir / "train.jsonl")
    write_jsonl([make_chat_example(record) for record in valid_records], output_dir / "valid.jsonl")

    return {
        "name": name,
        "output_dir": output_dir,
        "train_records": len(train_records),
        "valid_records": len(valid_records),
    }

In [59]:
summaries = [build_lora_dataset(name, paths) for name, paths in DATASETS.items()]
summaries

[{'name': 'formula_direct',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/formula_direct'),
  'train_records': 480,
  'valid_records': 240},
 {'name': 'algorithmic_scaffold',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/algorithmic_scaffold'),
  'train_records': 3200,
  'valid_records': 240},
 {'name': 'algorithmic_scaffold_v2',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/algorithmic_scaffold_v2'),
  'train_records': 4400,
  'valid_records': 240},
 {'name': 'algorithmic_scaffold_v2_5',
  'output_dir': PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/training_eval/fine_tune_qwen1_7B/lora/data/algorithmic_scaffold_v2_5'),
  'train_records': 4400,
  'valid_records': 240},
 {'name': 'algorithmic_scaffold_v3',
  'output_dir': Posi

In [60]:
example_path = OUTPUT_ROOT / "algorithmic_scaffold_v3_5" / "train.jsonl"
json.loads(example_path.read_text().splitlines()[0])


{'messages': [{'content': 'You solve discrete stochastic-process problems. Give concise reasoning, then end with exactly one final answer block: Final answer:\n<answer>\n{...}\n</answer>. Do not write anything after </answer>.',
   'role': 'system'},
  {'content': 'Let (S_n) be a simple symmetric random walk on the integers with S_0 = 8. Let tau = inf{n >= 0 : S_n in {0, 12}}. Compute E[tau]. Answer with JSON of the form {"expected_time": "..."} inside the answer tags.',
   'role': 'user'},
  {'content': 'Step 1: Identify the walk type.\nThe walk is simple symmetric, so p = q = 1/2. Use the symmetric finite-interval formula.\n\nStep 2: Extract the interval parameters.\nLower boundary a = 0; upper boundary b = 12; start x = 8.\n\nStep 3: Select the formula.\nFor the symmetric walk, E_x[tau] = (x-a)(b-x).\n\nStep 4: Compute each factor.\nx-a = 8 - (0) = 8.\nb-x = 12 - (8) = 4.\n\nStep 5: Multiply.\nE_x[tau] = 8 * 4 = 32.\n\nFinal answer:\n<answer>\n{"expected_time": "32"}\n</answer>',
  